In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [17]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['caseyyoungflesh_post', 'jackwasey_icd', 'oxfordcontrol_cosmo.jl', 'jbei_edd',
            'davidcarslaw_openair', 'qutech-delft_quantuminspire', 'tryolabs_norfair',
            'soft-matter_trackpy', 'cosmoscout_cosmoscout-vr', 'scify_jedaitoolkit']

list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0043f9cf35898788d2fe031c895320f5a95c4b5d,caseyyoungflesh_post


In [18]:
df.groupby('project').size()

project
caseyyoungflesh_post            724
cosmoscout_cosmoscout-vr       5442
davidcarslaw_openair           2017
jackwasey_icd                  3147
jbei_edd                       4862
oxfordcontrol_cosmo.jl         1308
qutech-delft_quantuminspire    1381
scify_jedaitoolkit              658
soft-matter_trackpy            2427
tryolabs_norfair               1452
dtype: int64

In [19]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  4%|▍         | 91/2342 [01:36<39:38,  1.06s/it]

Got Errors {'0f62e31e28f9b68ee357be68b987cb36205a36d2': 'Key 0f62e31e28f9b68ee357be68b987cb36205a36d2 not found in /da5_fast/All.sha1c/commit_15.tch'}


  4%|▍         | 96/2342 [01:41<39:39,  1.06s/it]

Got Errors {'13541202791bbd8c5dc81cc0f0ccec35fdb5248d': 'Key 13541202791bbd8c5dc81cc0f0ccec35fdb5248d not found in /da5_fast/All.sha1c/commit_19.tch'}


  5%|▌         | 122/2342 [02:08<39:05,  1.06s/it]

...

Got Errors {'fc00bb099866c0f85b84a704b2bb3f4cc085dd2d': 'Key fc00bb099866c0f85b84a704b2bb3f4cc085dd2d not found in /da5_fast/All.sha1c/commit_124.tch'}


 74%|███████▍  | 1731/2342 [30:39<10:51,  1.07s/it]

Got Errors {'ff158fcc755b4abe33fefec164c034d2ed8c25fe': 'Key ff158fcc755b4abe33fefec164c034d2ed8c25fe not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|██████████| 2342/2342 [41:35<00:00,  1.07s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0043f9cf35898788d2fe031c895320f5a95c4b5d,42c154315c9e5ced937c5987a96d4b47a4edd014,"[5b0b7ffff9efba768cd1b507985b3b97dba306bf, 0d7...",Casey Youngflesh <caseyyoungflesh@gmail.com>,1557198520,-0400,Casey Youngflesh <caseyyoungflesh@gmail.com>,1557198520,-0400,Merge branch 'master' of github.com:caseyyoung...,0043f9cf35898788d2fe031c895320f5a95c4b5d,caseyyoungflesh_post
1,01539dd71fa1e93fa064cf775bcb4e2d02f54f55,efe9557bb3c7cec64fe966d53cc65f1d8da4b099,[24555af068e0428f41235e5b1411f78fb8b1ecf1],Casey Youngflesh <caseyyoungflesh@gmail.com>,1512592109,-0500,Casey Youngflesh <caseyyoungflesh@gmail.com>,1512592109,-0500,Fix doc notes\n,01539dd71fa1e93fa064cf775bcb4e2d02f54f55,caseyyoungflesh_post


In [20]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0043f9cf35898788d2fe031c895320f5a95c4b5d', 'tree': '42c154315c9e5ced937c5987a96d4b47a4edd014', 'parent': ['5b0b7ffff9efba768cd1b507985b3b97dba306bf', '0d78e011872fb95d9f39f86c552b944a70191270'], 'author': 'Casey Youngflesh <caseyyoungflesh@gmail.com>', 'author_time': 1557198520, 'author_tz': '-0400', 'committer': 'Casey Youngflesh <caseyyoungflesh@gmail.com>', 'committer_time': 1557198520, 'committer_tz': '-0400', 'message': "Merge branch 'master' of github.com:caseyyoungflesh/MCMCvis\n"}


In [21]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].rename(columns={'author_time': 'time'})

In [22]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,caseyyoungflesh_post,0043f9cf35898788d2fe031c895320f5a95c4b5d,Casey Youngflesh <caseyyoungflesh@gmail.com>,1557198520,Merge branch 'master' of github.com:caseyyoung...


In [23]:
#mode a means append, so you have all your projects in the same file
yournetid='ccolli78'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='w', header=False)

**Notes.** 23,418 commit IDs found; 41 (0.2%) had no commit content in WoC and were skipped, leaving 23,377. WoC counts are higher than GitHub's because WoC includes all branches and forks. Data runs to 2026-04, later than the Jan 2025 cutoff in the README. Two repos were renamed: `caseyyoungflesh/post` → `MCMCvis`, `davidcarslaw/openair` → `openair-project/openair`.


In [24]:
missing = df[~df['sha1'].isin(df_commit_data['commit'])]
print(len(missing), 'commits could not be retrieved')
print(dfinf['project'].value_counts())


41 commits could not be retrieved
project
cosmoscout_cosmoscout-vr       5442
jbei_edd                       4855
jackwasey_icd                  3138
soft-matter_trackpy            2421
davidcarslaw_openair           2014
tryolabs_norfair               1452
qutech-delft_quantuminspire    1377
oxfordcontrol_cosmo.jl         1296
caseyyoungflesh_post            724
scify_jedaitoolkit              658
Name: count, dtype: int64


In [25]:
stats = dfinf.groupby('project').agg(ncommits=('commit', 'count'),
                                     nauthors=('author', 'nunique'),
                                     mintime=('time', 'min'),
                                     maxtime=('time', 'max'))
stats


,ncommits,nauthors,mintime,maxtime
project,,,,
caseyyoungflesh_post,724,8,1458715949,1624558029
cosmoscout_cosmoscout-vr,5442,67,1551714942,1775135410
davidcarslaw_openair,2014,35,1421337704,1762281919
jackwasey_icd,3138,36,1401671991,1734714091
jbei_edd,4855,28,1418339301,1733890409
oxfordcontrol_cosmo.jl,1296,31,1514998395,1758823682
qutech-delft_quantuminspire,1377,50,1532959943,1762546596
scify_jedaitoolkit,658,29,1458130700,1752295364
soft-matter_trackpy,2421,68,1340299131,1762541155


| Project | Commits | Authors | First | Last (WoC) | Stars | Forks | Last GH commit |
|---|---|---|---|---|---|---|---|
| caseyyoungflesh_post | 724 | 8 | 2016-03-23 | 2021-06-24 | 39 | 4 | 2025-11-25 |
| cosmoscout_cosmoscout-vr | 5,442 | 67 | 2019-03-04 | 2026-04-02 | 428 | 45 | 2026-09-07 |
| davidcarslaw_openair | 2,014 | 35 | 2015-01-15 | 2025-11-04 | 365 | 123 | 2026-09-28 |
| jackwasey_icd | 3,138 | 36 | 2014-06-02 | 2024-12-20 | 259 | 63 | 2026-09-14 |
| jbei_edd | 4,855 | 28 | 2014-12-11 | 2024-12-11 | 16 | 12 | 2025-01-09 |
| oxfordcontrol_cosmo.jl | 1,296 | 31 | 2018-01-03 | 2025-09-25 | 319 | 46 | 2026-07-07 |
| qutech-delft_quantuminspire | 1,377 | 50 | 2018-07-30 | 2025-11-07 | 75 | 30 | 2026-09-25 |
| scify_jedaitoolkit | 658 | 29 | 2016-03-16 | 2025-07-12 | 226 | 43 | 2025-07-12 |
| soft-matter_trackpy | 2,421 | 68 | 2012-06-21 | 2025-11-07 | 510 | 149 | 2026-06-29 |
| tryolabs_norfair | 1,452 | 65 | 2020-07-01 | 2025-09-17 | 2,688 | 277 | 2025-04-30 |

Commits/authors/dates from WoC; stars, forks and last commit from GitHub on 2026-09-30.

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github